---
# 🟢 PARTIE EXTRACT

### Ex E1 ⭐ — Explorer les sources
Affichez pour chaque DataFrame : shape, colonnes, dtypes, et les 3 premières lignes.

In [56]:

import numpy as np
import pandas as pd
from pandas.core.dtypes import dtypes
import os
import sqlite3
from path import Path
from pyspark.sql import SparkSession


In [57]:
# TON CODE ICI
#clients datafile
data_client = pd.read_csv(r"D:\python_projects\sales\clients.csv")
data_product = pd.read_csv(r"D:\python_projects\sales\produits.csv")
data_q1 = pd.read_csv(r"D:\python_projects\sales\commandes_Q1.csv")
data_q2 = pd.read_csv(r"D:\python_projects\sales\commandes_Q2.csv")


In [58]:
# TON CODE ICI
#ffichez pour chaque DataFrame : shape, colonnes, dtypes, et les 3 premières lignes
def audit_dataframe(df, name):
    print(f"Audit du DataFrame : {name}")
    print(f"Shape : {df.shape}")
    print(f"Colonnes : {df.columns.tolist()}")
    print(f"Dtypes :\n{df.dtypes}")
    print(f"3 premières lignes :\n{df.head(3)}")
    return df.shape, df.columns.tolist(), df.dtypes, df.head(3)
audit_dataframe(data_client, "Dataset Clients")

Audit du DataFrame : Dataset Clients
Shape : (515, 9)
Colonnes : ['client_id', 'nom', 'email', 'ville', 'region', 'segment', 'date_inscription', 'ca_historique', 'actif']
Dtypes :
client_id               str
nom                     str
email                   str
ville                   str
region                  str
segment                 str
date_inscription        str
ca_historique       float64
actif                  bool
dtype: object
3 premières lignes :
  client_id       nom             email     ville              region  \
0   CLI0001  Client_1  client1@email.fr  Toulouse         Auvergne-RA   
1   CLI0002  Client_2  client2@email.fr      Lyon  Nouvelle-Aquitaine   
2   CLI0003  Client_3  client3@email.fr    Nantes           Occitanie   

       segment date_inscription  ca_historique  actif  
0      Startup       2021-02-01         129.67   True  
1  Particulier       2022-08-11       42431.54  False  
2          PME       2023-11-17        6895.77   True  


((515, 9),
 ['client_id',
  'nom',
  'email',
  'ville',
  'region',
  'segment',
  'date_inscription',
  'ca_historique',
  'actif'],
 client_id               str
 nom                     str
 email                   str
 ville                   str
 region                  str
 segment                 str
 date_inscription        str
 ca_historique       float64
 actif                  bool
 dtype: object,
   client_id       nom             email     ville              region  \
 0   CLI0001  Client_1  client1@email.fr  Toulouse         Auvergne-RA   
 1   CLI0002  Client_2  client2@email.fr      Lyon  Nouvelle-Aquitaine   
 2   CLI0003  Client_3  client3@email.fr    Nantes           Occitanie   
 
        segment date_inscription  ca_historique  actif  
 0      Startup       2021-02-01         129.67   True  
 1  Particulier       2022-08-11       42431.54  False  
 2          PME       2023-11-17        6895.77   True  )

### Ex E2 ⭐ — Détecter les différences entre Q1 et Q2
Trouvez les colonnes présentes dans Q2 mais absentes de Q1, et vice-versa.
Affichez les différences clairement.

In [59]:
# Trouvez les colonnes présentes dans Q2 mais absentes de Q1, et vice-versa.Affichez les différences clairement
def compare_columns(df1, df2, name1, name2):
    colonnes_df1 = set(df1.columns)
    colonnes_df2 = set(df2.columns)

    colonnes_in_df1_not_in_df2 = colonnes_df1 - colonnes_df2
    colonnes_in_df2_not_in_df1 = colonnes_df2 - colonnes_df1

    print(f"Colonnes dans {name1} mais pas dans {name2}: {colonnes_in_df1_not_in_df2}")
    print(f"Colonnes dans {name2} mais pas dans {name1}: {colonnes_in_df2_not_in_df1}")
compare_columns(data_q1, data_q2, "commandes_Q1", "commandes_Q2")

Colonnes dans commandes_Q1 mais pas dans commandes_Q2: {'mode_paiement'}
Colonnes dans commandes_Q2 mais pas dans commandes_Q1: {'devise', 'moyen_paiement'}


### Ex E3 ⭐ — Harmoniser les colonnes
1. Renommez la colonne problématique dans Q2 pour qu'elle corresponde à Q1
2. Ajoutez la colonne manquante dans Q1 avec une valeur par défaut
3. Vérifiez que les deux DataFrames ont maintenant les mêmes colonnes

In [60]:

# Renommez la colonne problématique dans Q2 pour qu'elle corresponde à Q2
def rename_column(df, old_name, new_name):
    if old_name in df.columns:
        df.rename(columns={old_name: new_name}, inplace=True)
        print(f"Colonne renommée de '{old_name}' à '{new_name}'")
    else:
        print(f"Colonne '{old_name}' non trouvée dans le DataFrame.")
    return df.head(3)
rename_column(data_q2, "moyen_paiement", "mode_paiement")
        

Colonne renommée de 'moyen_paiement' à 'mode_paiement'


,commande_id,client_id,produit_id,quantite,prix_unitaire,montant_total,date_commande,statut,mode_paiement,region_livraison,devise
0,CMD-Q2-00001,CLI0122,PRD0041,5,48.15,240.75,2024-05-14,retournée,CB,Auvergne-RA,EUR
1,CMD-Q2-00002,CLI0346,PRD0106,1,52.42,52.42,2024-04-18,livrée,CB,PACA,EUR
2,CMD-Q2-00003,CLI0282,PRD0021,8,78.18,625.44,2024-05-05,annulée,CB,Occitanie,EUR


Donc, le moyen_paiement et mode_paiement exigent d'harmoniser.

In [61]:

#Ajoutez la colonne manquante dans Q1 avec une valeur par défaut
def add_missing_column(df, column_name, default_value):
    if column_name not in df.columns:
        df[column_name] = default_value
        print(f"Colonne '{column_name}' ajoutée avec la valeur par défaut '{default_value}'")
    else:
        print(f"Colonne '{column_name}' existe déjà dans le DataFrame.")
    return df.head(3)
add_missing_column(data_q2, "devis", "EUR")

Colonne 'devis' ajoutée avec la valeur par défaut 'EUR'


,commande_id,client_id,produit_id,quantite,prix_unitaire,montant_total,date_commande,statut,mode_paiement,region_livraison,devise,devis
0,CMD-Q2-00001,CLI0122,PRD0041,5,48.15,240.75,2024-05-14,retournée,CB,Auvergne-RA,EUR,EUR
1,CMD-Q2-00002,CLI0346,PRD0106,1,52.42,52.42,2024-04-18,livrée,CB,PACA,EUR,EUR
2,CMD-Q2-00003,CLI0282,PRD0021,8,78.18,625.44,2024-05-05,annulée,CB,Occitanie,EUR,EUR


In [62]:
#Vérifiez que les deux DataFrames ont maintenant les mêmes colonnes
#from columnar import columnar
def compare_columns(df1, df2):
    list_column = [df1.columns.tolist(), df2.columns.tolist()]

    print(list_column)

compare_columns(data_q1, data_q2)


[['commande_id', 'client_id', 'produit_id', 'quantite', 'prix_unitaire', 'montant_total', 'date_commande', 'statut', 'mode_paiement', 'region_livraison'], ['commande_id', 'client_id', 'produit_id', 'quantite', 'prix_unitaire', 'montant_total', 'date_commande', 'statut', 'mode_paiement', 'region_livraison', 'devise', 'devis']]


In [63]:
def currency_devise(df, column_name):
    df["column_name"] = "EUR" #currency
    print(data_q1.head(3))
currency_devise(data_q1, "devise")

    commande_id client_id produit_id  quantite  prix_unitaire  montant_total  \
0  CMD-Q1-00001   CLI0496    PRD0081         1          49.36          49.36   
1  CMD-Q1-00002   CLI0305    PRD0058         5         118.62         593.10   
2  CMD-Q1-00003   CLI0044    PRD0128         7          99.94         699.58   

  date_commande   statut mode_paiement    region_livraison column_name  
0    2024-03-18  annulée            CB           Occitanie         EUR  
1    2024-01-23   livrée            CB           Occitanie         EUR  
2    2024-01-15   livrée        PayPal  Nouvelle-Aquitaine         EUR  


### Ex E4 ⭐ — Consolider avec pd.concat
1. Ajoutez une colonne `source` dans Q1 (`'Q1_2024'`) et Q2 (`'Q2_2024'`)
2. Concaténez Q1 et Q2 avec `pd.concat`
3. Vérifiez que le total de lignes est cohérent
4. Affichez la répartition des lignes par source

In [64]:
# TON CODE ICI
#ajoute le colonne
data_q1['source'] = 'Q1_2024'
data_q2['source'] = 'Q2_2024'

data_conc = [data_q1, data_q2]
# Concatenate row-wise and reset index
data_commandes = pd.concat(data_conc, ignore_index=True)
data_commandes.shape

(6240, 14)

In [65]:
dff = data_commandes['source'].value_counts()
dff

source
Q2_2024    3220
Q1_2024    3020
Name: count, dtype: int64

### Ex E5 ⭐⭐ — Fonction extract()
Encapsulez tout dans une fonction `extract(q1_path, q2_path, clients_path, produits_path) -> dict`
qui retourne un dictionnaire avec les 4 DataFrames harmonisés et consolidés.
La fonction doit afficher des logs à chaque étape.

In [66]:
# TON CODE ICI
def extract(q1_path, q2_path, client_path, produit_path)->dict:
    data_client = pd.read_csv(client_path)
    data_produit = pd.read_csv(produit_path)
    data_q1 = pd.read_csv(q1_path)
    data_q2 = pd.read_csv(q2_path)

    # Harmonize column names
    data_q2 = data_q2.rename(columns={"moyen_payement": "mode_paiement"})
    data_q1["devis"] = "EUR"  # currency

    # Add source column
    data_q1['source'] = 'Q1_2024'
    data_q2['source'] = 'Q2_2024'

    # Concatenate row-wise and reset index
    data_commandes = pd.concat([data_q1, data_q2], axis=0, ignore_index=True, sort=False)
    
    return data_commandes, data_client, data_produit
extract(r"D:\python_projects\sales\commandes_Q1.csv", r"D:\python_projects\sales\commandes_Q2.csv", r"D:\python_projects\sales\clients.csv", r"D:\python_projects\sales\produits.csv")
data_commandes, data_client, data_product = extract(r"D:\python_projects\sales\commandes_Q1.csv", r"D:\python_projects\sales\commandes_Q2.csv", r"D:\python_projects\sales\clients.csv", r"D:\python_projects\sales\produits.csv")
data_commandes.columns

Index(['commande_id', 'client_id', 'produit_id', 'quantite', 'prix_unitaire',
       'montant_total', 'date_commande', 'statut', 'mode_paiement',
       'region_livraison', 'devis', 'source', 'moyen_paiement', 'devise'],
      dtype='str')

---
# 🟡 PARTIE TRANSFORM

### Ex T1 ⭐ — Audit initial
Écris une fonction `audit_dataframe(df, nom)` qui affiche :
- Shape
- Nombre de NaN total et par colonne (en %)
- Nombre de doublons
- Types de chaque colonne

In [67]:
# TON CODE ICI
def audit_dataframe(df, nom="DataFrame"):
    print(f"Audit du {nom} \n")

    # Shape
    print("Shape :", df.shape, "\n")

    # NaN total
    total_nan = df.isna().sum().sum()
    print(f"Nombre total de NaN : {total_nan}\n")

    # NaN par colonne
    nan_par_col = df.isna().mean() * 100
    print("Pourcentage de NaN par colonne :")
    print(nan_par_col.round(2).astype(str) + " %")
    print()

    # Doublons
    doublons = df.duplicated().sum()
    print(f"Nombre de doublons : {doublons}\n")

    # Types des colonnes
    print("Types des colonnes :")
    print(df.dtypes)
    print("\n=== Fin de l'audit ===")
#Exemple data pour clients
audit_dataframe(data_commandes, "Dataset Clients")


Audit du Dataset Clients 

Shape : (6240, 14) 

Nombre total de NaN : 12659

Pourcentage de NaN par colonne :
commande_id          0.0 %
client_id            0.0 %
produit_id           0.0 %
quantite             0.0 %
prix_unitaire        0.0 %
montant_total       1.88 %
date_commande        0.0 %
statut              0.99 %
mode_paiement       51.6 %
region_livraison     0.0 %
devis               51.6 %
source               0.0 %
moyen_paiement      48.4 %
devise              48.4 %
dtype: str

Nombre de doublons : 40

Types des colonnes :
commande_id             str
client_id               str
produit_id              str
quantite              int64
prix_unitaire       float64
montant_total       float64
date_commande           str
statut                  str
mode_paiement           str
region_livraison        str
devis                   str
source                  str
moyen_paiement          str
devise                  str
dtype: object

=== Fin de l'audit ===


### Ex T2 ⭐ — Supprimer les doublons
Sur le DataFrame consolidé :
1. Détectez et affichez le nombre de doublons
2. Montrez 3 exemples de lignes dupliquées
3. Supprimez les doublons (gardez la première occurrence)
4. Vérifiez qu'il n'en reste plus

In [68]:
# 1. Détecter et afficher le nombre de doublons

def audit_dataframe(df, nom="DataFrame"):
    print(f"Audit du {nom} \n")

    nb_doublons = df.duplicated().sum()
    print(f"Nombre de doublons : {nb_doublons}")

    # 2. Montrer 3 exemples de lignes dupliquées
    print("\nExemples de lignes dupliquées :")
    print(df[df.duplicated()].head(3))

    # 3. supprimer les doublons (garder la première occurrence)
    df_sans_doublons = df.drop_duplicates(keep="first")

    # 4. Vérifier qu'il n'en reste plus
    nb_restants = df_sans_doublons.duplicated().sum()
    print(f"\nDoublons restants après nettoyage : {nb_restants}")
# TON CODE ICI
#Exemple data pour clients
audit_dataframe(data_commandes, "Dataset Clients")


Audit du Dataset Clients 

Nombre de doublons : 40

Exemples de lignes dupliquées :
       commande_id client_id produit_id  quantite  prix_unitaire  \
3000  CMD-Q1-02950   CLI0271    PRD0124         2          55.52   
3001  CMD-Q1-00072   CLI0001    PRD0027         7          45.84   
3002  CMD-Q1-00145   CLI0441    PRD0020         9         304.72   

      montant_total date_commande    statut mode_paiement region_livraison  \
3000         111.04    2024-02-27   annulée            CB    Île-de-France   
3001         320.88    2024-03-20  en_cours            CB    Île-de-France   
3002        2742.48    2024-02-19    livrée            CB    Île-de-France   

     devis   source moyen_paiement devise  
3000   EUR  Q1_2024            NaN    NaN  
3001   EUR  Q1_2024            NaN    NaN  
3002   EUR  Q1_2024            NaN    NaN  

Doublons restants après nettoyage : 0


### Ex T3 ⭐ — Corriger les types
1. Convertissez `date_commande` en datetime en gérant les formats incohérents
2. Extrayez : `annee`, `mois`, `trimestre`, `jour_semaine`
3. Convertissez `quantite` en numérique avec gestion des erreurs
4. Affichez les types avant et après

In [102]:
#Convertissez `date_commande` en datetime en gérant les formats incohérents
# TON CODE ICI
def audit_dataframe(df, nom="DataFrame"):
    # 1. Afficher les dtypes avant
    #print("Data set: {df}".format(df=df.head()))
    print("Types avant :\n", df.dtypes, "\n")

    # 2. Convertir date_commande en datetime (formats incohérents gérés)
    df["date_commande"] = pd.to_datetime(df["date_commande"], errors="coerce", dayfirst=False)

    # 3. Extraire les composantes temporelles
    df["annee"] = df["date_commande"].dt.year
    df["mois"] = df["date_commande"].dt.month
    df["trimestre"] = df["date_commande"].dt.quarter
    df["jour_semaine"] = df["date_commande"].dt.day_name()

    # 4. Convertir quantite en numérique
    df["quantite"] = pd.to_numeric(df["quantite"], errors="coerce")

    # 5. Afficher les types APRÈS
    print("Types après :\n", df.dtypes)
audit_dataframe(data_commandes, "Dataset commande-Q1 et Q2")

Types avant :
 commande_id                              str
client_id                                str
produit_id                               str
quantite                               int64
prix_unitaire                        float64
montant_total                        float64
date_commande                 datetime64[us]
statut                                   str
mode_paiement                            str
region_livraison                         str
devis                                    str
source                                   str
moyen_paiement                           str
devise                                   str
annee                                float64
mois                                 float64
trimestre                            float64
jour_semaine                             str
montant_total_calculé                float64
tranche_montant                     category
saison                                   str
est_annulee                             

### Ex T4 ⭐⭐ — Traiter les NaN
Pour chaque colonne avec des NaN, choisissez et appliquez la stratégie adaptée :
- `montant_total` NaN → recalculer depuis `quantite × prix_unitaire`
- `statut` NaN → valeur par défaut `'inconnu'`
- `date_commande` NaT → supprimer la ligne (donnée critique)

Justifiez chaque choix dans un commentaire.

In [70]:
# TON CODE ICI
def audit_dataframe(df, nom="DataFrame"):
    #Total Missing values before population
    #montant_total NaN recalculer depuis quantite * prix_unitaire

    # NaN total before population
    print(f"Missing values before:, {df["montant_total"].isnull().sum()}\n")

    #populate missing values
    df["montant_total"] = df["quantite"]*df["prix_unitaire"]
    total_nan_after = df["montant_total"].isnull().sum()
    print(f"Missing values after population: {total_nan_after}\n")

    #statut NaN valeur par défault 'inconnu'
    df["statut"] = df["statut"].fillna("inconnu")
    print(df["statut"].isnull().sum())

    #date_commande Nat - supprimer la ligne

    print(f"Statut par défault inconnu: {df["montant_total"].fillna(df["montant_total"]).isnull().sum()}")
audit_dataframe(data_commandes, "dataset commande_Q1")

Missing values before:, 117

Missing values after population: 0

0
Statut par défault inconnu: 0


### Ex T5 ⭐⭐ — Détecter et traiter les aberrants
1. Détectez les prix négatifs → supprimez ces lignes
2. Détectez les quantités nulles ou négatives → supprimez
3. Vérifiez la cohérence `montant_total = quantite × prix_unitaire` → corrigez les incohérences
4. Affichez un résumé des corrections effectuées

In [71]:
# TON CODE ICI
#Detecter les prix negatifs et supprimer les lignes correspondantes
def detect_negative_prices(df):
    # Detecter les prix négatifs
    negative_prices = df[df['prix_unitaire'] < 0]
    print(f"Nombre de lignes avec prix négatif : {len(negative_prices)}")
    
    # Afficher les lignes avec prix négatif
    print("Exemples de lignes avec prix négatif :")
    print(negative_prices.head(3))
    
    # Supprimer les lignes avec prix négatif
    df_cleaned = df[df['prix_unitaire'] >= 0]
    
    return df_cleaned.head(3)
detect_negative_prices(data_commandes)

Nombre de lignes avec prix négatif : 22
Exemples de lignes avec prix négatif :
       commande_id client_id produit_id  quantite  prix_unitaire  \
348   CMD-Q1-00349   CLI0409    PRD0080         8        -131.51   
607   CMD-Q1-00608   CLI0330    PRD0028         2         -59.70   
1087  CMD-Q1-01088   CLI0219    PRD0057         7         -85.50   

      montant_total date_commande  statut mode_paiement region_livraison  \
348        -1052.08    2024-03-27  livrée        Chèque             PACA   
607         -119.40    2024-03-26  livrée        PayPal             PACA   
1087        -598.50    2024-02-26  livrée        PayPal             PACA   

     devis   source moyen_paiement devise   annee  mois  trimestre  \
348    EUR  Q1_2024            NaN    NaN  2024.0   3.0        1.0   
607    EUR  Q1_2024            NaN    NaN  2024.0   3.0        1.0   
1087   EUR  Q1_2024            NaN    NaN  2024.0   2.0        1.0   

     jour_semaine  
348     Wednesday  
607       Tuesday  
10

,commande_id,client_id,produit_id,quantite,prix_unitaire,montant_total,date_commande,statut,mode_paiement,region_livraison,devis,source,moyen_paiement,devise,annee,mois,trimestre,jour_semaine
0,CMD-Q1-00001,CLI0496,PRD0081,1,49.36,49.36,2024-03-18,annulée,CB,Occitanie,EUR,Q1_2024,NaN,NaN,2024.0,3.0,1.0,Monday
1,CMD-Q1-00002,CLI0305,PRD0058,5,118.62,593.10,2024-01-23,livrée,CB,Occitanie,EUR,Q1_2024,NaN,NaN,2024.0,1.0,1.0,Tuesday
2,CMD-Q1-00003,CLI0044,PRD0128,7,99.94,699.58,2024-01-15,livrée,PayPal,Nouvelle-Aquitaine,EUR,Q1_2024,NaN,NaN,2024.0,1.0,1.0,Monday


In [72]:
# detecter les quantités null ou négatives et supprimer les lignes correspondantes
def detect_invalid_quantities(df):
    # Detecter les quantités nulles ou négatives
    invalid_quantities = df[df['quantite'] <= 0]
    print(f"Nombre de lignes avec quantités nulles ou négatives : {len(invalid_quantities)}")
    
    # Afficher les lignes avec quantités nulles ou négatives
    print("Exemples de lignes avec quantités nulles ou négatives :")
    print(invalid_quantities.head(3))
    
    # Supprimer les lignes avec quantités nulles ou négatives
    df_cleaned = df[df['quantite'] > 0]
    
    return df_cleaned.head(3)
detect_invalid_quantities(data_commandes)

Nombre de lignes avec quantités nulles ou négatives : 31
Exemples de lignes avec quantités nulles ou négatives :
      commande_id client_id produit_id  quantite  prix_unitaire  \
202  CMD-Q1-00203   CLI0371    PRD0070         0          28.37   
366  CMD-Q1-00367   CLI0310    PRD0144         0         212.05   
630  CMD-Q1-00631   CLI0117    PRD0043         0         100.34   

     montant_total date_commande    statut mode_paiement region_livraison  \
202            0.0    2024-03-28    livrée            CB        Occitanie   
366            0.0    2024-01-25  en_cours        PayPal        Occitanie   
630            0.0    2024-01-21    livrée            CB    Île-de-France   

    devis   source moyen_paiement devise   annee  mois  trimestre jour_semaine  
202   EUR  Q1_2024            NaN    NaN  2024.0   3.0        1.0     Thursday  
366   EUR  Q1_2024            NaN    NaN  2024.0   1.0        1.0     Thursday  
630   EUR  Q1_2024            NaN    NaN  2024.0   1.0        1.0 

,commande_id,client_id,produit_id,quantite,prix_unitaire,montant_total,date_commande,statut,mode_paiement,region_livraison,devis,source,moyen_paiement,devise,annee,mois,trimestre,jour_semaine
0,CMD-Q1-00001,CLI0496,PRD0081,1,49.36,49.36,2024-03-18,annulée,CB,Occitanie,EUR,Q1_2024,NaN,NaN,2024.0,3.0,1.0,Monday
1,CMD-Q1-00002,CLI0305,PRD0058,5,118.62,593.10,2024-01-23,livrée,CB,Occitanie,EUR,Q1_2024,NaN,NaN,2024.0,1.0,1.0,Tuesday
2,CMD-Q1-00003,CLI0044,PRD0128,7,99.94,699.58,2024-01-15,livrée,PayPal,Nouvelle-Aquitaine,EUR,Q1_2024,NaN,NaN,2024.0,1.0,1.0,Monday


In [73]:
#Vérifiez la cohérence `montant_total = quantite × prix_unitaire` → corrigez les incohérences
def verify_consistency(df):
    # Calculer le montant total basé sur les colonnes existantes
    df['montant_total_calculé'] = df['quantite'] * df['prix_unitaire']
    
    # Identifier les incohérences
    df_inconsistent_rows = df[abs(df['montant_total'] - df['montant_total_calculé']) > 0]  # Tolérance pour les erreurs d arrondi
    print(f"Nombre de lignes avec incohérence : {len(df_inconsistent_rows)}")
    
    # Afficher les lignes avec incohérence
    print("Exemples de lignes avec incohérence :")
    print(df_inconsistent_rows.head(3))
    
    # Corriger les incohérences (optionnel)
    df.loc[df_inconsistent_rows.index, 'montant_total'] = df.loc[df_inconsistent_rows.index, 'montant_total_calculé']
    
    return df.head(3)
verify_consistency(data_commandes)

Nombre de lignes avec incohérence : 0
Exemples de lignes avec incohérence :
Empty DataFrame
Columns: [commande_id, client_id, produit_id, quantite, prix_unitaire, montant_total, date_commande, statut, mode_paiement, region_livraison, devis, source, moyen_paiement, devise, annee, mois, trimestre, jour_semaine, montant_total_calculé]
Index: []


,commande_id,client_id,produit_id,quantite,prix_unitaire,montant_total,date_commande,statut,mode_paiement,region_livraison,devis,source,moyen_paiement,devise,annee,mois,trimestre,jour_semaine,montant_total_calculé
0,CMD-Q1-00001,CLI0496,PRD0081,1,49.36,49.36,2024-03-18,annulée,CB,Occitanie,EUR,Q1_2024,NaN,NaN,2024.0,3.0,1.0,Monday,49.36
1,CMD-Q1-00002,CLI0305,PRD0058,5,118.62,593.10,2024-01-23,livrée,CB,Occitanie,EUR,Q1_2024,NaN,NaN,2024.0,1.0,1.0,Tuesday,593.10
2,CMD-Q1-00003,CLI0044,PRD0128,7,99.94,699.58,2024-01-15,livrée,PayPal,Nouvelle-Aquitaine,EUR,Q1_2024,NaN,NaN,2024.0,1.0,1.0,Monday,699.58


### Ex T6 ⭐⭐ — Jointures avec les référentiels
1. Nettoyez `df_clients` (doublons sur `client_id`)
2. Filtrez `df_produits` pour ne garder que les produits actifs
3. Faites un LEFT JOIN commandes ← clients (colonnes : ville, region, segment)
4. Faites un LEFT JOIN commandes ← produits (colonnes : nom, categorie, fournisseur)
5. Vérifiez le nombre de lignes avant et après (doit être identique avec LEFT JOIN)

In [74]:
# TON CODE ICI
#Nettoyez df_clients (doublons sur client_id)
def clean_clients(df):
    # Détecter les doublons sur client_id
    duplicates = df[df.duplicated(subset='client_id', keep=False)]
    print(f"Nombre de doublons sur client_id : {len(duplicates)}")
    
    # Afficher des exemples de doublons
    print("Exemples de doublons :")
    print(duplicates.head(3))
    
    # Supprimer les doublons en gardant la première occurrence
    df_cleaned = df.drop_duplicates(subset='client_id', keep='first')
    
    return df_cleaned
data_client_cleaned = clean_clients(data_client)

Nombre de doublons sur client_id : 30
Exemples de doublons :
   client_id        nom              email      ville         region  \
13   CLI0014  Client_14  client14@email.fr  Marseille      Occitanie   
32   CLI0033  Client_33  client33@email.fr     Nantes           PACA   
40   CLI0041  Client_41  client41@email.fr      Lille  Île-de-France   

         segment date_inscription  ca_historique  actif  
13       Startup       2022-06-01         317.70  False  
32       Startup       2020-12-02       10689.65   True  
40  Grand Compte       2020-10-03        1499.42  False  


In [75]:
#Filtrer df_produits pour ne garder que les produits actifs (statut = 'actif')
def filter_active_products(df):
    # Filtrer les produits actifs
    df_active = df[df['statut'] == 'actif']
    
    print(f"Nombre de produits actifs : {len(df_active)}")
    
    return df_active
filtered_products = filter_active_products(data_q2)


Nombre de produits actifs : 0


In [76]:
#Faites un LEFT JOIN commandes ← clients (colonnes : ville, region, segment)
def left_join_orders_clients(orders_df, clients_df):
    # Effectuer le LEFT JOIN sur la colonne 'client_id'
    merged_df = pd.merge(orders_df, clients_df[['client_id','ville', 'region', 'segment']], on='client_id', how='left')
    
    return merged_df.columns
left_join_orders_clients(data_commandes, data_client_cleaned)

Index(['commande_id', 'client_id', 'produit_id', 'quantite', 'prix_unitaire',
       'montant_total', 'date_commande', 'statut', 'mode_paiement',
       'region_livraison', 'devis', 'source', 'moyen_paiement', 'devise',
       'annee', 'mois', 'trimestre', 'jour_semaine', 'montant_total_calculé',
       'ville', 'region', 'segment'],
      dtype='str')

In [77]:
#Vérifiez le nombre de lignes avant et après (doit être identique avec LEFT JOIN)
print("Nombre de lignes dans data_q1:", len(data_commandes))
print("Nombre de lignes dans data_q2:", len(data_commandes))

Nombre de lignes dans data_q1: 6240
Nombre de lignes dans data_q2: 6240


### Ex T7 ⭐⭐ — Feature Engineering
Créez les colonnes suivantes :
1. `tranche_montant` : `<100€` / `100-500€` / `500-2000€` / `>2000€` avec `pd.cut()`
2. `saison` : depuis le mois (Hiver/Printemps/Été/Automne)
3. `est_annulee` : booléen True si statut == 'annulée'
4. `ca_perdu` : montant_total si commande annulée, sinon 0
5. `delai_inscription_commande` : nb de jours entre date_inscription du client et date_commande

Affichez `value_counts()` pour les colonnes catégorielles créées.

In [78]:
# TON CODE ICI
#`tranche_montant` : `<100€` / `100-500€` / `500-2000€` / `>2000€` avec `pd.cut()`
def create_tranche_montant(df):
    bins = [-np.inf, 100, 500, 2000, np.inf]
    labels = ['<100€', '100-500€', '500-2000€', '>2000€']
    df['tranche_montant'] = pd.cut(df['montant_total'], bins=bins, labels=labels)
    
    return df.head(3)
create_tranche_montant(data_commandes)

,commande_id,client_id,produit_id,quantite,prix_unitaire,montant_total,date_commande,statut,mode_paiement,region_livraison,devis,source,moyen_paiement,devise,annee,mois,trimestre,jour_semaine,montant_total_calculé,tranche_montant
0,CMD-Q1-00001,CLI0496,PRD0081,1,49.36,49.36,2024-03-18,annulée,CB,Occitanie,EUR,Q1_2024,NaN,NaN,2024.0,3.0,1.0,Monday,49.36,<100€
1,CMD-Q1-00002,CLI0305,PRD0058,5,118.62,593.10,2024-01-23,livrée,CB,Occitanie,EUR,Q1_2024,NaN,NaN,2024.0,1.0,1.0,Tuesday,593.10,500-2000€
2,CMD-Q1-00003,CLI0044,PRD0128,7,99.94,699.58,2024-01-15,livrée,PayPal,Nouvelle-Aquitaine,EUR,Q1_2024,NaN,NaN,2024.0,1.0,1.0,Monday,699.58,500-2000€


In [79]:
def create_saison(df):
    # Extraire le mois à partir de la colonne 'date_commande'
    df['mois'] = df['date_commande'].dt.month
    
    # Créer une fonction pour mapper les mois aux saisons
    def mois_to_saison(mois):
        if mois in [12, 1, 2]:
            return 'Hiver'
        elif mois in [3, 4, 5]:
            return 'Printemps'
        elif mois in [6, 7, 8]:
            return 'Été'
        elif mois in [9, 10, 11]:
            return 'Automne'
        else:
            return np.nan  # Cas où le mois est invalide
    
    # Appliquer la fonction pour créer la colonne 'saison'
    df['saison'] = df['mois'].apply(mois_to_saison)
    
    return df.head(3)
create_saison(data_commandes)

,commande_id,client_id,produit_id,quantite,prix_unitaire,montant_total,date_commande,statut,mode_paiement,region_livraison,...,source,moyen_paiement,devise,annee,mois,trimestre,jour_semaine,montant_total_calculé,tranche_montant,saison
0,CMD-Q1-00001,CLI0496,PRD0081,1,49.36,49.36,2024-03-18,annulée,CB,Occitanie,...,Q1_2024,NaN,NaN,2024.0,3.0,1.0,Monday,49.36,<100€,Printemps
1,CMD-Q1-00002,CLI0305,PRD0058,5,118.62,593.10,2024-01-23,livrée,CB,Occitanie,...,Q1_2024,NaN,NaN,2024.0,1.0,1.0,Tuesday,593.10,500-2000€,Hiver
2,CMD-Q1-00003,CLI0044,PRD0128,7,99.94,699.58,2024-01-15,livrée,PayPal,Nouvelle-Aquitaine,...,Q1_2024,NaN,NaN,2024.0,1.0,1.0,Monday,699.58,500-2000€,Hiver


In [80]:
# `est_annulee` : booléen True si statut == 'annulée'
def create_est_annulee(df):
    df['est_annulee'] = df['statut'] == 'annulée'
    return df.head(3)
create_est_annulee(data_commandes)


,commande_id,client_id,produit_id,quantite,prix_unitaire,montant_total,date_commande,statut,mode_paiement,region_livraison,...,moyen_paiement,devise,annee,mois,trimestre,jour_semaine,montant_total_calculé,tranche_montant,saison,est_annulee
0,CMD-Q1-00001,CLI0496,PRD0081,1,49.36,49.36,2024-03-18,annulée,CB,Occitanie,...,NaN,NaN,2024.0,3.0,1.0,Monday,49.36,<100€,Printemps,True
1,CMD-Q1-00002,CLI0305,PRD0058,5,118.62,593.10,2024-01-23,livrée,CB,Occitanie,...,NaN,NaN,2024.0,1.0,1.0,Tuesday,593.10,500-2000€,Hiver,False
2,CMD-Q1-00003,CLI0044,PRD0128,7,99.94,699.58,2024-01-15,livrée,PayPal,Nouvelle-Aquitaine,...,NaN,NaN,2024.0,1.0,1.0,Monday,699.58,500-2000€,Hiver,False


In [81]:
# `ca_perdu` : montant_total si commande annulée, sinon 0
def create_ca_perdu(df):
    df['ca_perdu'] = np.where(df['est_annulee'], df['montant_total'], 0)
    return df.head(3)
create_ca_perdu(data_commandes)

,commande_id,client_id,produit_id,quantite,prix_unitaire,montant_total,date_commande,statut,mode_paiement,region_livraison,...,devise,annee,mois,trimestre,jour_semaine,montant_total_calculé,tranche_montant,saison,est_annulee,ca_perdu
0,CMD-Q1-00001,CLI0496,PRD0081,1,49.36,49.36,2024-03-18,annulée,CB,Occitanie,...,NaN,2024.0,3.0,1.0,Monday,49.36,<100€,Printemps,True,49.36
1,CMD-Q1-00002,CLI0305,PRD0058,5,118.62,593.10,2024-01-23,livrée,CB,Occitanie,...,NaN,2024.0,1.0,1.0,Tuesday,593.10,500-2000€,Hiver,False,0.00
2,CMD-Q1-00003,CLI0044,PRD0128,7,99.94,699.58,2024-01-15,livrée,PayPal,Nouvelle-Aquitaine,...,NaN,2024.0,1.0,1.0,Monday,699.58,500-2000€,Hiver,False,0.00


In [82]:
# `delai_inscription_commande` : nb de jours entre date_inscription du client et date_commande
def create_delai_inscription_commande(df1, df2): #values for date_inscription and date_commande from respective dataframes
    # Convertir les colonnes en datetime si ce n'est pas déjà fait
    df1['date_inscription'] = pd.to_datetime(df1['date_inscription'], errors='coerce')
    df2['date_commande'] = pd.to_datetime(df2['date_commande'], errors='coerce')
    
    # Calculer le délai en jours
    df2['delai_inscription_commande'] = (df2['date_commande'] - df1['date_inscription']).dt.days #column added in Q1-2024 and Q2-2024 dataframes
    
    return df2.head(3)
create_delai_inscription_commande(data_client_cleaned, data_commandes)

,commande_id,client_id,produit_id,quantite,prix_unitaire,montant_total,date_commande,statut,mode_paiement,region_livraison,...,annee,mois,trimestre,jour_semaine,montant_total_calculé,tranche_montant,saison,est_annulee,ca_perdu,delai_inscription_commande
0,CMD-Q1-00001,CLI0496,PRD0081,1,49.36,49.36,2024-03-18,annulée,CB,Occitanie,...,2024.0,3.0,1.0,Monday,49.36,<100€,Printemps,True,49.36,1141.0
1,CMD-Q1-00002,CLI0305,PRD0058,5,118.62,593.10,2024-01-23,livrée,CB,Occitanie,...,2024.0,1.0,1.0,Tuesday,593.10,500-2000€,Hiver,False,0.00,530.0
2,CMD-Q1-00003,CLI0044,PRD0128,7,99.94,699.58,2024-01-15,livrée,PayPal,Nouvelle-Aquitaine,...,2024.0,1.0,1.0,Monday,699.58,500-2000€,Hiver,False,0.00,59.0


### Ex T8 ⭐⭐⭐ — Validation qualité
Écris une fonction `valider_pipeline(df) -> bool` qui :
1. Vérifie qu'il n'y a plus de NaN sur les colonnes critiques
2. Vérifie qu'il n'y a plus de doublons
3. Vérifie que tous les prix sont > 0
4. Vérifie que toutes les quantités sont > 0
5. Vérifie que montant_total == quantite × prix_unitaire (à 1 centime près)
6. Retourne True si tout est OK, False sinon avec les erreurs détaillées

In [83]:
# TON CODE ICI
#La fonction qui valider_pipeline(df) -> booléen True si toutes les validations sont passées, sinon False
#Verifier qu'il n'y a plus de doublons
def valider_pipeline(df):
    # Vérifier les doublons
    if df.duplicated().any():
        print("Échec de la validation : des doublons sont présents.")
        return False
    
    # Vérifier les prix négatifs
    if (df['prix_unitaire'] < 0).any():
        print("Échec de la validation : des prix négatifs sont présents.")
        return False
    
    # Vérifier les quantités nulles ou négatives
    if (df['quantite'] <= 0).any():
        print("Échec de la validation : des quantités nulles ou négatives sont présentes.")
        return False
    
    # Vérifier la cohérence du montant total
    incoherences = df[abs(df['montant_total'] - (df['quantite'] * df['prix_unitaire'])) > 0]
    if not incoherences.empty:
        print("Échec de la validation : des incohérences dans le montant total sont présentes.")
        return False
    
    print("Toutes les validations sont passées avec succès.")
    return True
valider_pipeline(data_commandes)

Échec de la validation : des doublons sont présents.


False

### Ex T9 ⭐⭐⭐ — Analyse métier post-transform
Maintenant que les données sont propres, répondez en code :
1. Quel est le CA total du S1 2024 ?
2. Quel trimestre a le CA le plus élevé ?
3. Quelle région génère le plus de CA ?
4. Quel segment client est le plus rentable ?
5. Quelle catégorie de produit a le taux d'annulation le plus élevé ?
6. Quel est le CA perdu sur les annulations ?

In [84]:
# TON CODE ICI
#Maintenant que les données sont propres, répondez en code :
    
    
# quel est le CA total du S1 2024 ?

def calculate_ca_total_s1_2024(df):
    # Filtrer les commandes du S1 2024 (janvier à juin)
    df_s1_2024 = df[(df['date_commande'].dt.year == 2024) & (df['date_commande'].dt.month <= 6)]
    
    # Calculer le CA total
    ca_total = df_s1_2024['montant_total'].sum()
    
    return ca_total
ca_total_s1_2024 = calculate_ca_total_s1_2024(data_commandes)
print(f"CA total du S1 2024 : {ca_total_s1_2024} €")

CA total du S1 2024 : 4003250.24 €


In [85]:
# Quel trimestre a le CA le plus élevé ?
def calculate_highest_ca_quarter(df):
    # Grouper par année et trimestre et calculer le CA total
    ca_par_trimestre = df.groupby(['annee', 'trimestre'])['montant_total'].sum().reset_index()
    
    # Trouver le trimestre avec le CA le plus élevé
    highest_ca_row = ca_par_trimestre.loc[ca_par_trimestre['montant_total'].idxmax()]
    
    return highest_ca_row
ca_highest_ca_quarter = calculate_highest_ca_quarter(data_commandes)
ca_highest_ca_quarter


annee               2024.00
trimestre              2.00
montant_total    2075683.84
Name: 1, dtype: float64

In [86]:
# Laquelle région génère le plus de CA
def calculate_highest_ca_region(df):
    # Grouper par région et calculer le CA total
    ca_par_region = df.groupby('region_livraison')['montant_total'].sum().reset_index()
    
    # Trouver la région avec le CA le plus élevé
    highest_ca_region_row = ca_par_region.loc[ca_par_region['montant_total'].idxmax()]
    
    return highest_ca_region_row
ca_highest_ca_region = calculate_highest_ca_region(data_commandes)
ca_highest_ca_region

region_livraison    Occitanie
montant_total       853720.79
Name: 2, dtype: object

In [87]:
data_client.columns

Index(['client_id', 'nom', 'email', 'ville', 'region', 'segment',
       'date_inscription', 'ca_historique', 'actif'],
      dtype='str')

In [88]:
# Le segment client est le plus rentable ?
def calculate_most_profitable_segment(df1, df2):
    # Effectuer un LEFT JOIN entre les commandes et les clients pour obtenir le segment
    merged_df = df1.merge(df2, on='client_id', how='left')
    # Grouper par segment et calculer le CA total
    ca_par_segment = merged_df.groupby('segment')['montant_total'].sum().reset_index()
    
    # Trouver le segment avec le CA le plus élevé
    most_profitable_segment_row = ca_par_segment.loc[ca_par_segment['montant_total'].idxmax()]
    
    return most_profitable_segment_row
most_profitable_segment = calculate_most_profitable_segment(data_commandes, data_client)
most_profitable_segment

segment          Particulier
montant_total     2169749.17
Name: 2, dtype: object

In [89]:
# La catégorie de produit a le taux d'annulation le plus élevé ?
def calculate_highest_cancellation_rate_category(df1, df2):
    # Effectuer un LEFT JOIN entre les commandes et les produits pour obtenir la catégorie
    merged_df = df1.merge(df2, on='produit_id', how='left')
    
    # Grouper par catégorie et calculer le taux d'annulation
    cancellation_rate_par_categorie = merged_df.groupby('categorie').apply(
        lambda x: (x['est_annulee'].sum() / len(x)) * 100
    ).reset_index(name='taux_annulation')
    
    # Trouver la catégorie avec le taux d'annulation le plus élevé
    highest_cancellation_rate_row = cancellation_rate_par_categorie.loc[
        cancellation_rate_par_categorie['taux_annulation'].idxmax()
    ]
    
    return highest_cancellation_rate_row
highest_cancellation_rate_category = calculate_highest_cancellation_rate_category(data_commandes, data_product)
highest_cancellation_rate_category

categorie             Réseau
taux_annulation    10.942761
Name: 3, dtype: object

In [90]:
# Le CA perdu sur les annulations
def calculate_ca_perdu(df):
    # Filtrer les commandes annulées
    df_annulees = df[df['est_annulee']]
    
    # Calculer le CA perdu
    ca_perdu = df_annulees['montant_total'].sum()
    
    return ca_perdu
ca_perdu = calculate_ca_perdu(data_commandes)
print(f"CA perdu sur les annulations : {ca_perdu} €")


CA perdu sur les annulations : 407965.98 €


In [91]:
#data_commandes.columns

---
# 🔵 PARTIE LOAD

### Ex L1 ⭐ — Export CSV
1. Exportez le DataFrame enrichi en CSV dans `../output/commandes_enrichies.csv`
2. Vérifiez la taille du fichier en Ko
3. Rechargez le CSV et vérifiez que le shape est identique

In [92]:

#Exportez le DataFrame enrichi en CSV dans `../output/commandes_enrichies.csv`

from pathlib import Path
def export_csv(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)  # crée les dossiers
    df.to_csv(path, index=False, encoding="utf-8")  # écrase si existe
    print(f" Export OK : {path}")
    
    #Taille de fichier csv
    file_size_csv = os.path.getsize(path)/1024
    print(file_size_csv)
    
export_csv(data_commandes, ".Output/commandes_enrichies.csv")


 Export OK : .Output\commandes_enrichies.csv
975.109375


### Ex L2 ⭐⭐ — Export Parquet
1. Exportez en Parquet dans `../output/commandes_S1.parquet`
2. Comparez la taille CSV vs Parquet
3. Rechargez le Parquet et vérifiez le shape
4. Mesurez le temps de lecture CSV vs Parquet avec `%%time`

In [93]:


#Output en csv ../Output/commandes_S1.csv

def export_parquet(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)  # crée les dossiers
    df.to_parquet(path, index=False)  # écrase si existe
    print(f"Export OK : {path}")
    
    file_size_parquet = os.path.getsize(path)/1024
    print(file_size_parquet)
    #Export
export_parquet(data_commandes, ".output/commandes_S1.parquet")

Export OK : .output\commandes_S1.parquet
264.37890625


In [94]:
#compare la taille de fichier en csv et parquet

def compare_taille_fichier(path1, path2):
    
    
    file_size_csv = Path(path1)
    file_size_parquet = Path(path2)
    
    file_size_csv = os.path.getsize(path1)/1024
    #print(file_size1)
    file_size_parquet = os.path.getsize(path2)/1024
    #print(file_size2)
    
    if file_size_csv > file_size_parquet:
        print(f"Size of csv file is larger in size compared to parquet file")
    elif file_size_csv < file_size_parquet:
        print(f"cvs file is smaller than parquet")
    else:
        print("They have the same size")
        
compare_taille_fichier('.Output/commandes_enrichies.csv', '.Output/commandes_S1.parquet')

Size of csv file is larger in size compared to parquet file


### Ex L3 ⭐⭐ — SQLite
1. Créez une base SQLite `../output/ecommerce.db`
2. Chargez le DataFrame dans une table `commandes`
3. Créez une deuxième table `reporting` avec un agrégat par trimestre et région
4. Exécutez au moins 3 requêtes SQL directement depuis Python et affichez les résultats

In [95]:
from pathlib import Path

def export_sql(df, path):

    def connect_sqlite(path):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)  # crée les dossiers

    conn = sqlite3.connect(".Output/cecommerce.db")
    
    #charger le fiche en données
    def export_sqlite(df, path):
        df.to_sql(path, con=conn, if_exists="replace")
        conn.commit()
        conn.close()

    #export des données en db sqlite
export_sql(data_commandes, ".Output/ecommerce.db")
    

### Ex L4 ⭐⭐⭐ — Parquet partitionné
Exportez le DataFrame en Parquet partitionné par `trimestre` et `region_livraison`.

Puis répondez :
- Combien de fichiers Parquet sont créés ?
- Quelle est la taille totale vs le Parquet non partitionné ?
- Quel est l'avantage du partitionnement pour PySpark ?

In [96]:
#from pyspark.context import SparkContext


In [97]:

# TON CODE ICI
#Dataframe en Parquet partitionné par trimestre et region_livraison
# Create Spark session
# creating sparksession and giving an app name


#park = SparkSession.builder.appName('sparkdf').getOrCreate()

---
# 🔴 PIPELINE COMPLET

### Ex P1 ⭐⭐ — Pipeline modulaire
Écris 3 fonctions séparées :
- `extract(data_dir) -> dict`
- `transform(sources: dict) -> pd.DataFrame`
- `load(df, output_dir) -> None`

Chacune doit avoir une docstring et des logs.
Teste chaque fonction indépendamment.

In [103]:

# TON CODE ICI
#Pipeline
# fonction extract(data_dir) ->dict

#data auditing
def audit_dataframe(df, name):
    print(f"Audit du DataFrame : {name}")
    print(f"Shape : {df.shape}")
    print(f"Colonnes : {df.columns.tolist()}")
    print(f"Duplicates: {df.duplicates().sum()}")
    print(f"Total NaN: {df.isna().sum().sum()}")
    print(f"Nan % per column: {(df.isna().mean()*100).round(2)}")
    print(f"Dtypes :\n{df.dtypes}")
    
def compare_columns(df1, df2, name1, name2):
    colonnes_df1 = set(df1.columns)
    colonnes_df2 = set(df2.columns)

    colonnes_in_df1_not_in_df2 = colonnes_df1 - colonnes_df2
    colonnes_in_df2_not_in_df1 = colonnes_df2 - colonnes_df1

    print(f"Colonnes dans {name1} mais pas dans {name2}: {colonnes_in_df1_not_in_df2}")
    print(f"Colonnes dans {name2} mais pas dans {name1}: {colonnes_in_df2_not_in_df1}")

def rename_column(df, old_name, new_name):
    if old_name in df.columns:
        df.rename(columns={old_name: new_name}, inplace=True)
        print(f"Colonne renommée de '{old_name}' à '{new_name}'")
    else:
        print(f"Colonne '{old_name}' non trouvée dans le DataFrame.")
    return df.head(3)

def add_missing_column(df, column_name, default_value):
    if column_name not in df.columns:
        df[column_name] = default_value
        print(f"Colonne '{column_name}' ajoutée avec la valeur par défaut '{default_value}'")
    else:
        print(f"Colonne '{column_name}' existe déjà dans le DataFrame.")
    return df.head(3)

def compare_columns(df1, df2):
    list_column = [df1.columns.tolist(), df2.columns.tolist()]

    print(list_column)
    


### Ex P2 ⭐⭐⭐ — Pipeline avec gestion d'erreurs
Améliore ton pipeline avec :
1. `try/except` sur chaque étape critique
2. Un log d'erreur détaillé si une étape échoue
3. La pipeline ne doit pas crasher complètement si un fichier est manquant — elle doit logger l'erreur et continuer avec les autres

In [99]:
# TON CODE ICI


### Ex P3 ⭐⭐⭐ — Tests unitaires du pipeline
Écris 5 tests pour ton pipeline :
1. `test_extract_retourne_4_sources()`
2. `test_transform_supprime_doublons()`
3. `test_transform_pas_de_prix_negatif()`
4. `test_transform_montant_coherent()`
5. `test_load_cree_les_fichiers()`

Chaque test affiche ✅ PASS ou ❌ FAIL.

In [100]:
# TON CODE ICI


### Ex P4 ⭐⭐⭐⭐ — PROJET FINAL : Pipeline automatisé
Crée un pipeline complet qui :

1. **Détecte automatiquement** tous les fichiers CSV dans `../data_etl/` qui commencent par `commandes_`
2. **Les traite tous** sans avoir à spécifier les chemins manuellement
3. **Génère un rapport HTML** avec :
   - Les statistiques clés (CA, nb commandes, taux annulation)
   - 3 graphiques Plotly intégrés
4. **Sauvegarde** en CSV + Parquet + SQLite
5. **Loggue** chaque étape avec timestamps

C'est exactement ce que vous ferez en PySpark la semaine prochaine, mais à l'échelle de plusieurs To.

📎 Docs utiles :
- [glob — trouver des fichiers](https://docs.python.org/3/library/glob.html)
- [logging](https://docs.python.org/3/library/logging.html)
- [Plotly — export HTML](https://plotly.com/python/interactive-html-export/)


In [101]:
# TON CODE ICI
